# Build churn features

Materializes the feature frame consumed by the scoring notebook. Runs cold top-to-bottom.

In [1]:
import os
import pandas as pd
from churn_lib.io import read_events, write_features

In [2]:
scoring_date = None
lookback_days = 90
min_active_days = 14
feature_path = None

In [3]:
events = read_events(scoring_date=scoring_date, lookback_days=lookback_days)
events = events.astype({"account_id": "int64", "event_ts": "datetime64[ns]"})

In [4]:
active = events.groupby("account_id")["event_ts"].nunique().rename("active_days")
active = active[active >= min_active_days]

In [5]:
recency = (
    events.groupby("account_id")["event_ts"].max().rename("last_seen")
)
frequency = events.groupby("account_id").size().rename("event_count")

In [6]:
features = (
    pd.concat([active, recency, frequency], axis=1, join="inner")
    .reset_index()
)
features["scoring_date"] = scoring_date

In [7]:
write_features(features, feature_path)
print(f"wrote {len(features)} feature rows")